# Portfolio Construction

## Signal Evaluation

### Information Coefficient

The IC is the **cross-sectional correlation between your signal and subsequent realized returns**. At each rebalance date $t$:

$$
IC_t = \text{corr}\big(s_{i,t},\ r_{i,t+1}\big) \quad \text{across all assets } i
$$

- **Pearson IC** — ordinary linear correlation. Sensitive to outliers and to the (often non-linear) signal-return relationship. A few extreme returns can dominate it.

- **Rank IC (Spearman)** — correlation of the *ranks* of signal and return. This is the industry standard for signal evaluation, because what you usually care about is whether the signal *orders* assets correctly (top-ranked outperform bottom-ranked), not whether the relationship is linear. Rank IC is robust to outliers and to monotonic non-linearity.

#### IC Stability — the IC Information Ratio

Recall you have a time series of $IC_t$. The single most important quality metric is the ratio of its mean to its volatility: $\text{IC-IR} = \frac{\overline{IC}}{\sigma_{IC}}$. This is the consistency of your skill.

#### Hit Rate

What fraction of periods is $IC_t > 0$? Hit rate tells you whether the edge is broad-based across time or driven by a few lucky periods.

#### Monotonicity (quantile analysis)

Sort assets into quantiles (deciles) by signal, and check that realized returns increase monotonically across quantiles — top decile beats 9th beats 8th... beats bottom. A genuine signal is monotonic; a fake one might have a good top-minus-bottom spread but a jumbled middle, revealing the "signal" is really just identifying a few extreme names rather than ranking the whole universe.

#### Decay

Compute IC at multiple forward horizons — $\text{corr}(s_{i,t}, r_{i,t+1})$, $\text{corr}(s_{i,t}, r_{i,t+5})$, $\text{corr}(s_{i,t}, r_{i,t+20})$. This traces how fast the signal's predictive power decays.

Two archetypes, with opposite implementation demands:

- **Fast-decaying signal** — IC peaks at 1 day and is gone by 5. The information is short-lived (think order-flow imbalance, short-term reversal). To capture it you must trade *fast and often* — high turnover. The alpha is real but perishable and harder to transfer (turnover limits bite).
- **Slow-decaying signal** — IC builds over weeks and persists for months (think value, quality, slow fundamental signals). You can trade patiently, low turnover, and still capture most of the alpha.

##### Turnover

**Turnover** is how much of the portfolio you trade per period — the sum of absolute weight changes between rebalances:

$$
\text{Turnover} = \frac{1}{2}\sum_i |w_{i,t} - w_{i,t-1}|
$$

(The $\frac{1}{2}$ counts a round trip — selling one name to buy another — as one unit of turnover, so 100% turnover means you've replaced the whole book once.) Annualize it by multiplying by rebalances per year.

##### Half-life

Model the signal's autocorrelation as decaying exponentially — the half-life is how long until the signal retains half its predictive power. A 2-day half-life is a fast signal; a 3-month half-life is slow. This single number drives almost everything downstream:

- **Effective breadth** (from Grinold-Kahn) — a signal with a short half-life refreshes into genuinely *new* independent bets more often, raising breadth; but only if you can trade fast enough to act on each refresh. A 3-month-half-life signal rebalanced daily does *not* give you 252 independent bets per year — it gives you ~4, because consecutive observations are nearly the same bet.
- **Required trading frequency** — you must rebalance at a cadence matched to the decay. Trade too slowly and the signal has decayed before you act (you capture stale alpha); trade too fast relative to the decay and you're just churning on noise, paying costs for no incremental information.

##### Transaction Cost

**Transaction costs have two pieces**, and the distinction matters:

- **Linear / proportional costs** — bid-ask spread, commissions, fees. Cost scales linearly with the dollar amount traded: $c \cdot |\Delta w|$. Predictable, and crucially, *convex* — so they can be folded into the optimizer as a penalty without breaking the convex structure (a turnover or $\ell_1$ trade penalty, which stays QP/SOCP).
- **Market impact** — your own trading moves the price against you. This is *non-linear*, typically the *square-root* law: impact $\propto \sigma\sqrt{Q/V}$, where $Q$ is your order size, $V$ is daily volume, $\sigma$ is volatility. Trading more moves the price more, but sublinearly (the square root).

#### Net-of-cost Alpha

**The net-of-cost alpha — this is the punchline.** What you actually earn is gross alpha minus turnover times cost-per-trade:

$$
\text{Net alpha} \approx \text{Gross alpha} - (\text{Turnover} \times \text{Cost per unit traded})
$$

So a high-IC fast signal can have *negative net alpha* if its turnover is high enough that costs exceed the gross edge. This is exactly where paper alpha dies. The fast signals with the most theoretical breadth are the ones whose alpha is most likely eaten by costs.

### Capacity

**The drivers of capacity** — what makes a strategy high- or low-capacity:

- **Signal decay / turnover** — fast signals require high turnover, and high turnover means you trade more, hit impact more often, and exhaust capacity faster. A fast-decaying signal has *low* capacity. A slow signal you trade patiently has *high* capacity. This is the deep reason fast high-IC signals are often *less* valuable to a large fund than slow modest-IC ones — the fast one can't absorb size.
- **Liquidity of the universe** — capacity scales with the daily volume $V$ of the names you trade. A signal that works on large-cap, high-volume stocks (Russell 1000) has far more capacity than one that lives in small-caps.
- **Breadth** — more independent names means you spread the same capital across more positions, so each position is smaller relative to its stock's volume, so impact per name is lower. Breadth raises capacity (another reason breadth matters beyond the Grinold-Kahn IR).

### Information Ratio

#### The Fundamental Law of Active Management

The central result (Grinold, 1989):

$$
IR \approx IC \times \sqrt{BR}
$$

Your **information ratio** — risk-adjusted active return, the active-management analog of the Sharpe ratio — equals your **information coefficient** times the square root of **breadth**. Three terms:

- **IR (Information Ratio)** $= \frac{\alpha}{\omega}$ — active return $\alpha$ divided by active risk (tracking error) $\omega$. How much risk-adjusted outperformance you deliver versus the benchmark. An IR of 0.5 is good, 1.0 is excellent, sustained.
- **IC (Information Coefficient)** — the *correlation between your forecasts and the realized returns*. Your skill per bet. An IC of 0.05 means your predictions are very weakly but positively correlated with outcomes — and in this game, that's already meaningful. IC of 0.1 is strong.
- **BR (Breadth)** — the number of *independent* bets you make per year. 100 stocks you rebalance quarterly with independent views ≈ 400 independent bets.

Because breadth enters as $\sqrt{BR}$, you can achieve the same IR two completely different ways:

- **High skill, low breadth** — a concentrated manager with IC 0.1 making 16 independent bets a year: $IR = 0.1 \times \sqrt{16} = 0.4$.
- **Low skill, high breadth** — a systematic manager with IC 0.05 making 64 independent bets: $IR = 0.05 \times \sqrt{64} = 0.4$.

Same information ratio. Half the skill, compensated by four times the breadth. This is the mathematical justification for systematic, diversified, many-small-bets investing: you don't need to be brilliant on any single position if you can make many weakly-skilled independent bets. It's why quant equity works — modest per-stock skill, applied across hundreds of names, produces strong risk-adjusted returns.

**Why $\sqrt{BR}$ and not $BR$.**

This is the key subtlety. Independent bets diversify — their idiosyncratic errors partially cancel (the same diversification math from the covariance work). Doubling the number of independent bets doesn't double your IR; it multiplies it by $\sqrt{2}$, because risk reduction from diversification scales with the square root of the number of independent positions (standard error of a mean shrinks as $1/\sqrt{n}$). So breadth helps, but with diminishing returns — the square root is the diversification signature.

#### The Transfer Coefficient

The original $IR = IC \times \sqrt{BR}$ assumes you can build a portfolio that *perfectly* expresses your views — that the weights end up exactly proportional to your forecasts. In reality you can't, because constraints get in the way: long-only mandates, position limits, sector neutrality, turnover caps, transaction costs. Grinold and Kahn added a correction term to handle this — the **transfer coefficient**:

$$
IR \approx IC \times \sqrt{BR} \times TC
$$

**The transfer coefficient (TC)** is the *correlation between your forecasts and your actual portfolio positions*. It measures how faithfully your views get "transferred" into the portfolio:

- $TC = 1$ — perfect transfer; positions exactly reflect forecasts (the idealized, unconstrained case). This recovers the original law.
- $TC = 0.5$ — typical for a long-only constrained equity portfolio; constraints cut your realized IR in half.
- $TC \to 0$ — constraints so binding the portfolio barely reflects your views at all.

Why this matters enormously. The transfer coefficient is precisely where portfolio construction meets alpha. You can have great forecasts (high IC) and many bets (high BR), but if your construction process can't express those views — because no-shorting pins your negative views at zero, or turnover limits stop you trading, or sector neutrality cancels your tilts — then $TC$ is low and your realized IR collapses regardless of how good your signals were.

This is the quantitative bridge: the alpha side *generates* IC and BR; the construction side *determines* TC. A great optimizer with thoughtful constraints preserves TC; a clumsy one destroys it. Concretely a binding no-short constraint pins assets you'd want to short at zero. Each such binding constraint is the transfer coefficient leaking: a view you held that the portfolio couldn't express.

So the full picture:

$$
IR = \underbrace{IC}_{\text{skill (alpha)}} \times \underbrace{\sqrt{BR}}_{\text{diversification (alpha)}} \times \underbrace{TC}_{\text{construction fidelity}}
$$

Three levers: skill per bet, number of independent bets, and how cleanly the portfolio expresses them. The transfer coefficient is *why* portfolio construction matters to an alpha process — it's the multiplier that says good construction is worth as much as good signals.

### Cost-aware Rebalancing

You don't trade all the way to the "ideal" signal-implied portfolio every period, because the marginal alpha of the last bit of trading isn't worth its cost. Instead you trade *partway* — to the point where the marginal alpha gained equals the marginal cost incurred. Concretely, you add a transaction-cost penalty to the objective:

$$
\max_w \ \mu^Tw - \frac{\gamma}{2}w^T\Sigma w - \lambda\|w - w_{t-1}\|_1
$$

That last term — the $\ell_1$ penalty on the *trade* (the change from current holdings) — is the **no-trade band**: small signal changes don't justify trading at all, and you only rebalance when the signal has moved enough that the alpha exceeds the cost. This connects straight to the KKT/complementary-slackness machinery — the no-trade region is exactly where the trade constraint is slack. And it stays convex (the $\ell_1$ term is convex), so it's still a QP/SOCP — the same solver machinery, with one more penalty term.

Cost-aware rebalancing deliberately lowers your transfer coefficient — you're choosing not to fully express the signal because expressing it costs more than it's worth. So there's an optimal TC below 1: pushing TC toward 1 (trading all the way to the ideal portfolio) maximizes gross alpha but incurs maximum cost; the net-alpha-maximizing TC trades off the two. This is the precise, quantitative reason the transfer coefficient is rarely 1 in practice — it's not just constraints binding, it's the optimal response to costs.

### Signal Evaluation Pitfalls

1. **Lookahead bias — the most common and most fatal. Point-in-time data failures. Survivorship bias. The IC red flag:** an IC above ~0.15 in equities should trigger an immediate lookahead audit, not celebration. Real signals are weak; a strong one usually means information leaked.

2. **Overfitting / multiple testing.** Defenses: hold out a true out-of-sample period you *never* touch during development, penalize for the number of trials (the deflated Sharpe / deflated IC adjustments), and prefer signals with an *economic rationale* over pure data-mined ones.

3. **IC doesn't account for costs or capacity.** IC measures *predictive* skill; it says nothing about *implementable* skill.

4. **Regime dependence and non-stationarity.** Look at the IC *over time* (rolling windows).

5. **The cardinal rule — out-of-sample is the only honest test.**

## Signal Combination

**Diversification of signals — the same √ benefit as diversifying assets.** If two signals each have IC 0.05 and are *uncorrelated*, the combined signal has a higher IC-IR than either alone, because their idiosyncratic noise partially cancels — exactly the diversification math from the covariance work, now applied to forecasts instead of returns. The combined IC-IR scales roughly with √K for K independent signals of equal quality.

**The crucial dependency: signal correlation.** Just as with assets, the benefit depends on how *correlated* the signals are. Two signals that are 0.9 correlated are nearly the same bet — combining them adds little. Two uncorrelated signals each add genuinely new information. So the value of combination is driven by signal *independence*, not signal count.

**1. Equal weighting (the "1/N" combination).** Standardize each signal cross-sectionally (z-score it), then average:

$$
s_{\text{combined}} = \frac{1}{K}\sum_{k=1}^K z_k
$$

Crude, but shockingly hard to beat — and that's a genuine result, not a throwaway. The same reason 1/N is hard to beat in *asset* allocation applies here: estimating optimal combination weights requires estimating the signals' IC and covariance, which is noisy, and the estimation error often outweighs the benefit of departing from equal weights. So equal weighting is the robust default and the benchmark every fancier method must beat. For *uncorrelated, similar-quality* signals, it's close to optimal.

**2. IC-weighting.** Weight each signal by its information coefficient — trust the more skillful signals more:

$$
s_{\text{combined}} = \sum_k IC_k \cdot z_k
$$

This is the Grinold-Kahn idea applied to combination: a signal's weight should reflect its skill. Better than equal-weight when the signals genuinely differ in quality and you can estimate $IC_k$ reliably. The risk: $IC_k$ is itself noisy, so IC-weighting can overfit to which signal happened to look best in-sample.

**3. Optimal (mean-variance) signal weighting.** Treat the signals exactly like assets in a Markowitz problem. You want the combination weights $\mathbf{w}$ that maximize the *combined signal's* IC-IR, and the solution is — recognize it —

$$
\mathbf{w}^* \propto \Omega^{-1} \mathbf{IC}
$$

where $\mathbf{IC}$ is the vector of each signal's information coefficient and $\Omega$ is the **covariance matrix of the signals**. This is *literally Markowitz*: IC plays the role of expected returns, the signal covariance plays the role of $\Sigma$, and the optimal combination is $\Sigma^{-1}\mu$ in signal space. So combining signals optimally is the *same mathematical problem* as building an optimal portfolio — you've already solved it.

And it inherits *all* the same problems: $\Omega$ (signal covariance) is noisy and possibly ill-conditioned, so $\Omega^{-1}$ amplifies estimation error (the same $\Sigma^{-1}$ disease), the combination weights come out extreme and unstable (error-maximization, now on signals), and you need exactly the same fixes — **shrinkage of the signal covariance** (Ledoit-Wolf, again), or constraints on the combination weights. The entire portfolio-construction toolkit reappears, one level up, with signals as the "assets."

**4. Regression-based (and ML) combination.** Regress forward returns on all the signals jointly:

$$
r_{i,t+1} = \beta_1 s_{i,t}^{(1)} + \cdots + \beta_K s_{i,t}^{(K)} + \epsilon
$$

The fitted $\beta_k$ are the combination weights, automatically accounting for signal correlation (a signal that's redundant given others gets a low coefficient). This is the cleanest framing — but raw OLS overfits with many correlated signals, so in practice you regularize: **ridge** (shrink coefficients) or **LASSO** (shrink and select). Ridge regression here is *exactly* the shrinkage idea again — the same $\beta = (X^TX + \lambda I)^{-1}X^Ty$ that connects to covariance ridge-regularization. Tree ensembles (gradient boosting) and other ML methods are the non-linear generalization, capturing signal *interactions*, at the cost of more overfitting risk and less interpretability.

Signal combination is portfolio construction with signals as assets — same optimal formula ($\Omega^{-1}\mathbf{IC}$), same overfitting disease, same fixes (shrinkage, constraints, out-of-sample discipline, humility about regimes) — and equal-weighting is the robust default that fancier methods must beat.

### The Pitfalls of Combination

1. Overfitting the combination weights. This is the dominant danger. With $K$ signals you're estimating $K$ combination weights (plus their covariance), and each is noisy. The "optimal" $\Omega^{-1}\mathbf{IC}$ weights are fit on historical data and will look spectacular in-sample and disappoint out-of-sample — the exact error-maximization disease, now on signals. The defenses are the same ones: shrink toward equal weights (the 1/N prior), constrain the weights, or regularize the regression. The blunt truth: a well-chosen equal-weight combination usually beats an "optimally" estimated one out-of-sample, for the same reason 1/N beats estimated-Markowitz.

2. Correlated signals masquerading as breadth. If you combine ten signals that are all variants of momentum, you think you have breadth ten but you really have breadth one. The combination's diversification benefit is illusory, and you'll overestimate the resulting IC-IR badly. Always look at the signal correlation matrix before combining — and the eigenvalue structure of it (the RMT lens again): if one big eigenvalue dominates, your "many signals" are really one factor in disguise. Genuine breadth requires genuinely independent information sources.

3. Look-ahead in the combination step. Subtle and lethal: if you choose the combination weights (or even which signals to include) using the full sample, you've leaked future information into the backtest. The IC-weights and the signal covariance must be estimated using only data available at each point in time — rolling or expanding windows, never the full history. This is the lookahead-bias pitfall from yesterday, now hiding in the meta-layer of how you combined rather than in the signals themselves. It's especially easy to miss because it doesn't feel like using future returns — but choosing weights with hindsight is exactly that.

4. Regime instability of the combination. The optimal mix of signals changes over time — momentum works in trending regimes, value in mean-reverting ones, sentiment in certain conditions. A combination optimized on one regime can be wrong in the next. Static combination weights assume a stationarity that markets don't honor. This is why adaptive/time-varying combination (or just robust equal-weighting) is often preferred over a single "optimal" historical fit — and why some shops use regime-aware combination, which is sophisticated but adds its own overfitting surface.